In [1]:
import pandas as pd
import numpy as np
import os

## Data validations and constraint checks

In [3]:
data_dir = "../data"
files = {
    "sorties": os.path.join(data_dir, "sorties.csv"),
    "aircraft": os.path.join(data_dir, "aircraft.csv"),
    "cadets": os.path.join(data_dir, "cadets.csv"),
    "instructors": os.path.join(data_dir, "instructors.csv"),
    "toga_study": os.path.join(data_dir, "toga_study.csv"),
    "payments": os.path.join(data_dir, "payments.csv")
}

dfs = {}
for name, path in files.items():
    if os.path.exists(path):
        dfs[name] = pd.read_csv(path)
    else:
        print(f"Warning: File {path} not found. Creating mock DataFrame for validation demo.")
        dfs[name] = pd.DataFrame()

print("--- INITIALIZING DATA VALIDATION SUITE ---")

def check_invalid_dates(df, date_cols):
    errors = 0
    for col in date_cols:
        if col in df.columns:
        
            filled_dates = df[col].dropna().astype(str).str.strip()
            filled_dates = filled_dates[filled_dates != ""]
            
            converted = pd.to_datetime(filled_dates, errors='coerce')
            invalid_count = converted.isna().sum()
            errors += invalid_count
    return errors

report = []

for name, df in dfs.items():
    if df.empty:
        continue
        
    metrics = {
        "Dataset": name,
        "Total Rows": len(df),
        "Missing Values (Total Cells)": df.isna().sum().sum() + (df == "").sum().sum(),
        "Duplicate IDs": 0,
        "Invalid Dates": 0,
        "Negative Values": 0,
        "Logical Anomalies Detected": 0
    }
    

    id_cols = [c for c in df.columns if "id" in c and not c.startswith("home") and not c.startswith("base")]
    if id_cols and name != "toga_study" and name != "sorties": 
    
        metrics["Duplicate IDs"] = df.duplicated(subset=[id_cols[0]]).sum()
    elif name == "toga_study":
        metrics["Duplicate IDs"] = df.duplicated(subset=["cadet_id", "subject"]).sum()
        

    date_cols = [c for c in df.columns if "date" in c or "start" in c or "end" in c]
    metrics["Invalid Dates"] = check_invalid_dates(df, date_cols)
    

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        metrics["Negative Values"] += (df[col] < 0).sum()
        
    report.append(metrics)

report_df = pd.DataFrame(report)
print("\n--- SURFACE-LEVEL QUALITY METRICS ---")
print(report_df.to_string(index=False))

print("\n--- RUNNING DEEP LOGICAL ANOMALY CHECKS ---\n")

if "sorties" in dfs:
    invalid_status = dfs["sorties"][~dfs["sorties"]["status"].isin(["completed", "cancelled"])]
    if len(invalid_status) > 0:
        print(f"[FAIL] Sorties: Found {len(invalid_status)} rows with invalid status codes.")
    else:
        print("[PASS] Sorties: All status entries are structurally valid ('completed' or 'cancelled').")


    s_df = dfs["sorties"].copy()
    s_df["actual_start_dt"] = pd.to_datetime(s_df["actual_start"], errors='coerce')
    s_df["actual_end_dt"] = pd.to_datetime(s_df["actual_end"], errors='coerce')
    
    time_anomalies = s_df[s_df["status"] == "completed"]
    bad_times = time_anomalies[time_anomalies["actual_start_dt"] >= time_anomalies["actual_end_dt"]]
    if len(bad_times) > 0:
        print(f"[FAIL] Sorties: Found {len(bad_times)} completed flights where start time is after/equals end time.")
    else:
        print("[PASS] Sorties: Flight times are chronologically sequential for completed entries.")

if "payments" in dfs:
    p_df = dfs["payments"].copy()
    p_df.replace("", np.nan, inplace=True)
    p_df = p_df.fillna(0)
    

    calc_mismatch = (p_df["invoiced_amount"] != (p_df["paid_amount"] + p_df["outstanding_amount"])).sum()
    if calc_mismatch > 0:
        print(f"[FAIL] Payments: Found {calc_mismatch} entries where Ledger formula breaks (Invoiced != Paid + Outstanding).")
    else:
        print("[PASS] Payments: Financial ledger statements balance perfectly.")

if "toga_study" in dfs:
    ts_df = dfs["toga_study"].copy()
    ts_df.replace("", np.nan, inplace=True)
    


    bad_progress = ts_df[pd.to_numeric(ts_df["chapters_completed"]) > pd.to_numeric(ts_df["total_chapters"])]
    bad_scores = ts_df[(pd.to_numeric(ts_df["avg_quiz_score"]) > 100) | (pd.to_numeric(ts_df["avg_quiz_score"]) < 0)]
    
    if len(bad_progress) > 0 or len(bad_scores) > 0:
        print(f"[FAIL] TOGA Study: Found {len(bad_progress)} rows with chapters over total, and {len(bad_scores)} rows with out-of-bounds quiz scores.")
    else:
        print("[PASS] TOGA Study: Academic metrics, chapter limits, and exam bounds are valid.")

if "aircraft" in dfs:
    ac_df = dfs["aircraft"].copy()
    ac_df.replace("", np.nan, inplace=True)
    
    over_maintenance = ac_df[pd.to_numeric(ac_df["maintenance_downtime_hours"]) > pd.to_numeric(ac_df["total_available_hours"])]
    if len(over_maintenance) > 0:
        print(f"[FAIL] Aircraft: Found {len(over_maintenance)} airplanes with more downtime hours than total available operational hours.")
    else:
        print("[PASS] Aircraft: Maintenance clocks do not exceed structural limits.")

if "cadets" in dfs:
    c_df = dfs["cadets"].copy()
    c_df.replace("", np.nan, inplace=True)
    
    overflight = c_df[pd.to_numeric(c_df["total_flown_hours"]) > pd.to_numeric(c_df["total_required_hours"])]
    if len(overflight) > 0:
        print(f"[FAIL] Cadets: Detected {len(overflight)} students who logged more flight hours than their syllabus requirement.")
    else:
        print("[PASS] Cadets: Student flight times align with standard program requirements.")

--- INITIALIZING DATA VALIDATION SUITE ---

--- SURFACE-LEVEL QUALITY METRICS ---
    Dataset  Total Rows  Missing Values (Total Cells)  Duplicate IDs  Invalid Dates  Negative Values  Logical Anomalies Detected
    sorties         100                           115              0              0                0                           0
   aircraft          10                             4              0              0                0                           0
     cadets          20                             1              0              0                0                           0
instructors          10                             1              0              0                0                           0
 toga_study          50                             9              0              0                0                           0
   payments          20                             6              0              0                0                           0

--- RUNNING DE

In [6]:
print("=" * 80)
print("             ADVANCED CROSS-DOMAIN DATA VALIDATION ENGINE             ")
print("=" * 80)

print("\n[+] RUNNING FLIGHT SORTIE INTEGRITY CHECKS...")

s_df = dfs["sorties"].copy()
s_df["sched_start_dt"] = pd.to_datetime(s_df["scheduled_start"])
s_df["actual_start_dt"] = pd.to_datetime(s_df["actual_start"])
s_df["actual_end_dt"] = pd.to_datetime(s_df["actual_end"])

missing_times_completed = s_df[(s_df["status"] == "completed") & (s_df["actual_start"].isna() | s_df["actual_end"].isna())]
if not missing_times_completed.empty:
    print(f"  └─ [CRITICAL FAIL] Found {len(missing_times_completed)} completed sorties missing actual execution timestamps.")
else:
    print("  └─ [PASS] Completed sorties have valid execution timestamps.")

cancelled_with_times = s_df[(s_df["status"] == "cancelled") & (s_df["actual_start"].notna() | s_df["actual_end"].notna())]
if not cancelled_with_times.empty:
    print(f"  └─ [CRITICAL FAIL] Found {len(cancelled_with_times)} cancelled sorties that incorrectly logged flight times.")
else:
    print("  └─ [PASS] Cancelled sorties contain no phantom flight times.")

s_df["calculated_delay"] = (s_df["actual_start_dt"] - s_df["sched_start_dt"]).dt.total_seconds() / 60.0
s_df["calculated_delay"] = s_df["calculated_delay"].fillna(0).astype(int)
s_df["delay_minutes"] = pd.to_numeric(s_df["delay_minutes"]).fillna(0).astype(int)

delay_mismatches = s_df[s_df["status"] == "completed"][s_df["delay_minutes"] != s_df["calculated_delay"]]
if not delay_mismatches.empty:
    print(f"  └─ [FAIL] Found {len(delay_mismatches)} sorties where logged delay minutes do not match timestamp differences.")
else:
    print("  └─ [PASS] Logged delay telemetry matches scheduled vs actual timestamps.")


print("\n[+] RUNNING FINANCIAL TRANSACTIONS AUDIT...")

p_df = dfs["payments"].copy()
p_df["invoiced_amount"] = pd.to_numeric(p_df["invoiced_amount"]).fillna(0)
p_df["paid_amount"] = pd.to_numeric(p_df["paid_amount"]).fillna(0)
p_df["outstanding_amount"] = pd.to_numeric(p_df["outstanding_amount"]).fillna(0)

ledger_mismatch = p_df[p_df["outstanding_amount"] != (p_df["invoiced_amount"] - p_df["paid_amount"])]
if not ledger_mismatch.empty:
    print(f"  └─ [CRITICAL FAIL] Ledger imbalance found in {len(ledger_mismatch)} cadet financial records.")
else:
    print("  └─ [PASS] Outstanding accounts equal invoiced amount minus paid amount across all ledgers.")


print("\n[+] RUNNING GROUND SCHOOL ACADEMIC INSPECTION...")

ts_df = dfs["toga_study"].copy()
ts_df["chapters_completed"] = pd.to_numeric(ts_df["chapters_completed"]).fillna(0)
ts_df["total_chapters"] = pd.to_numeric(ts_df["total_chapters"]).fillna(0)

impossible_progress = ts_df[ts_df["chapters_completed"] > ts_df["total_chapters"]]
if not impossible_progress.empty:
    print(f"  └─ [FAIL] Detected {len(impossible_progress)} rows where chapters completed exceeds total subject chapters.")
else:
    print("  └─ [PASS] Academic progression curves are bounded within realistic 100% limits.")

print("\n[+] RUNNING CROSS-DOMAIN CROSS-TABLE ANALYSIS...")

c_df = dfs["cadets"].copy()
c_df["total_flown_hours"] = pd.to_numeric(c_df["total_flown_hours"]).fillna(0)

flying_cadets = set(c_df[c_df["total_flown_hours"] > 0]["cadet_id"].unique())
studying_cadets = set(ts_df["cadet_id"].unique())

stalled_academics = flying_cadets - studying_cadets
if stalled_academics:
    print(f"  └─ [FAIL] Found {len(stalled_academics)} cadets flying sorties without any recorded ground school study activity.")
    print(f"     └─ Flagged Cadet IDs: {sorted(list(stalled_academics))}")
else:
    print("  └─ [PASS] All actively flying cadets have corresponding ground school study logs.")

print("\n[+] RUNNING FLEET LOGISTICS & MAINTENANCE AUDIT...")

ac_df = dfs["aircraft"].copy()
ac_df["defect_count"] = pd.to_numeric(ac_df["defect_count"]).fillna(0)

defect_threshold = ac_df["defect_count"].mean() + (2 * ac_df["defect_count"].std())
high_defects = ac_df[ac_df["defect_count"] > defect_threshold]
if not high_defects.empty:
    print(f"  └─ [WARNING] Identified {len(high_defects)} aircraft with unusually high defect counts (Threshold: {defect_threshold:.1f} defects).")
else:
    print("  └─ [PASS] Fleet defect distributions are within safe operational limits.")

print("\n" + "=" * 80)
print("                    VALIDATION RUN COMPLETE                         ")
print("=" * 80)

             ADVANCED CROSS-DOMAIN DATA VALIDATION ENGINE             

[+] RUNNING FLIGHT SORTIE INTEGRITY CHECKS...
  └─ [PASS] Completed sorties have valid execution timestamps.
  └─ [PASS] Cancelled sorties contain no phantom flight times.
  └─ [PASS] Logged delay telemetry matches scheduled vs actual timestamps.

[+] RUNNING FINANCIAL TRANSACTIONS AUDIT...
  └─ [PASS] Outstanding accounts equal invoiced amount minus paid amount across all ledgers.

[+] RUNNING GROUND SCHOOL ACADEMIC INSPECTION...
  └─ [PASS] Academic progression curves are bounded within realistic 100% limits.

[+] RUNNING CROSS-DOMAIN CROSS-TABLE ANALYSIS...
  └─ [FAIL] Found 1 cadets flying sorties without any recorded ground school study activity.
     └─ Flagged Cadet IDs: ['C002']

[+] RUNNING FLEET LOGISTICS & MAINTENANCE AUDIT...
  └─ [PASS] Fleet defect distributions are within safe operational limits.

                    VALIDATION RUN COMPLETE                         


C:\Users\abhim\AppData\Local\Temp\ipykernel_10124\2478881967.py:28: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  delay_mismatches = s_df[s_df["status"] == "completed"][s_df["delay_minutes"] != s_df["calculated_delay"]]


In [12]:
data_dir = "../data"
report_dir = "../reports"
os.makedirs(report_dir, exist_ok=True)

files = {
    "sorties": os.path.join(data_dir, "sorties.csv"),
    "aircraft": os.path.join(data_dir, "aircraft.csv"),
    "instructors": os.path.join(data_dir, "instructors.csv")
}

try:
    sorties = pd.read_csv(files["sorties"])
    aircraft = pd.read_csv(files["aircraft"])
    instructors = pd.read_csv(files["instructors"])
except FileNotFoundError as e:
    print(f"Error loading source data: {e}")
    print("Please ensure sorties.csv, aircraft.csv, and instructors.csv exist in ../data")
    exit(1)

sorties['actual_start'] = pd.to_datetime(sorties['actual_start'], errors='coerce')
sorties['actual_end'] = pd.to_datetime(sorties['actual_end'], errors='coerce')
sorties['scheduled_start'] = pd.to_datetime(sorties['scheduled_start'], errors='coerce')
sorties['scheduled_end'] = pd.to_datetime(sorties['scheduled_end'], errors='coerce')

aircraft['total_available_hours'] = pd.to_numeric(aircraft['total_available_hours'], errors='coerce').fillna(150.0)
aircraft['maintenance_downtime_hours'] = pd.to_numeric(aircraft['maintenance_downtime_hours'], errors='coerce').fillna(0.0)
aircraft['defect_count'] = pd.to_numeric(aircraft['defect_count'], errors='coerce').fillna(0.0)
instructors['total_duty_hours'] = pd.to_numeric(instructors['total_duty_hours'], errors='coerce').fillna(130.0)

sorties['flown_hours'] = (sorties['actual_end'] - sorties['actual_start']).dt.total_seconds() / 3600.0
sorties['flown_hours'] = sorties['flown_hours'].fillna(0.0)


ac_flown = sorties[sorties['status'] == 'completed'].groupby('aircraft_id')['flown_hours'].sum().reset_index()
aircraft = aircraft.merge(ac_flown, on='aircraft_id', how='left').fillna({'flown_hours': 0.0})
aircraft['utilization_rate'] = aircraft['flown_hours'] / aircraft['total_available_hours']

base_util = aircraft.groupby('base_id')[['flown_hours', 'total_available_hours']].sum()
base_util['utilization_rate'] = base_util['flown_hours'] / base_util['total_available_hours']

under_ac = aircraft[aircraft['utilization_rate'] < 0.05]['aircraft_id'].tolist()
high_defect_ac = aircraft[aircraft['defect_count'] >= 8]
operational_review_ac = aircraft[(aircraft['utilization_rate'] < 0.05) | (aircraft['defect_count'] >= 8)]['aircraft_id'].tolist()

inst_flown = sorties[sorties['status'] == 'completed'].groupby('instructor_id')['flown_hours'].sum().reset_index()
instructors = instructors.merge(inst_flown, on='instructor_id', how='left').fillna({'flown_hours': 0.0})

overloaded_inst = instructors[instructors['flown_hours'] > 15]['instructor_id'].tolist()
underutilized_inst = instructors[instructors['flown_hours'] < 10]['instructor_id'].tolist()

merged_sorties = sorties.merge(aircraft[['aircraft_id', 'type']], on='aircraft_id', how='left')
mismatch_count = 0

for idx, row in merged_sorties[merged_sorties['status'] == 'completed'].iterrows():
    inst_row = instructors[instructors['instructor_id'] == row['instructor_id']]
    if not inst_row.empty and pd.notna(inst_row.iloc[0]['aircraft_qualified']):
        allowed_types = [t.strip() for t in str(inst_row.iloc[0]['aircraft_qualified']).split(',')]
        if pd.notna(row['type']) and row['type'] not in allowed_types:
            mismatch_count += 1

total_sorties = len(sorties)
completed_sorties = len(sorties[sorties['status'] == 'completed'])
cancelled_sorties = len(sorties[sorties['status'] == 'cancelled'])
delayed_sorties = len(sorties[(sorties['status'] == 'completed') & (sorties['delay_minutes'] > 0)])

completion_rate = completed_sorties / total_sorties if total_sorties > 0 else 0
cancellation_rate = cancelled_sorties / total_sorties if total_sorties > 0 else 0
avg_delay = sorties[sorties['status'] == 'completed']['delay_minutes'].mean()

cancel_reasons = sorties[sorties['status'] == 'cancelled']['cancel_reason'].value_counts()

sorties['day'] = sorties['scheduled_start'].dt.strftime('%Y-%m-%d')
delay_by_day = sorties[sorties['status'] == 'completed'].groupby('day')['delay_minutes'].mean()
delay_by_base = sorties[sorties['status'] == 'completed'].groupby('base_id')['delay_minutes'].mean()
delay_by_lesson = sorties[sorties['status'] == 'completed'].groupby('lesson_type')['delay_minutes'].mean()

report_path = os.path.join(report_dir, "skynet_operations_analysis.md")

with open(report_path, "w") as f:
    f.write("# Skynet Operations & Efficiency Analysis Report\n")
    f.write(f"**Flight Training Organization (FTO) Performance Review** *Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
    f.write("---\n\n")


    f.write("## 1. Aircraft Utilization Analysis\n\n")
    f.write(" Aircraft-Wise Utilization Profile\n")
    f.write("| Aircraft ID | Registration | Type | Base | Flown Hours | Available Hours | Utilization Rate | Defect Count | Maintenance Downtime (Hrs) |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n")
    for _, row in aircraft.iterrows():
        f.write(f"| {row['aircraft_id']} | {row['registration']} | {row['type'] if pd.notna(row['type']) else 'N/A'} | {row['base_id'] if pd.notna(row['base_id']) else 'N/A'} | {row['flown_hours']:.2f} | {row['total_available_hours']:.1f} | {row['utilization_rate']*100:.1f}% | {int(row['defect_count'])} | {int(row['maintenance_downtime_hours'])} |\n")
    
    f.write("\n### Base-Wise Utilization Matrix\n")
    f.write("| Base ID | Combined Flown Hours | Combined Available Hours | Aggregate Utilization Rate |\n")
    f.write("| :--- | :--- | :--- | :--- |\n")
    for base, row in base_util.iterrows():
        if pd.notna(base) and base != "":
            f.write(f"| {base} | {row['flown_hours']:.2f} | {row['total_available_hours']:.1f} | {row['utilization_rate']*100:.1f}% |\n")

    f.write("\n### Operational Asset Flags\n")
    f.write(f"* **Underutilized Aircraft (<5.0% Efficiency Index):** {', '.join(under_ac) if under_ac else 'None flagged.'}\n")
    f.write(f"* **Aircraft with High Defect Counts (>= 8 Incidents):** {', '.join([f'{r.aircraft_id} ({int(r.defect_count)} defects)' for _, r in high_defect_ac.iterrows()]) if not high_defect_ac.empty else 'None flagged.'}\n")
    f.write(f"* **Maintenance Downtime Impact:** The fleet suffered a total loss of **{int(aircraft['maintenance_downtime_hours'].sum())} hours** due to reactive technical maintenance.\n")
    f.write(f"* **Aircraft Flagged for Operational Review:** {', '.join(operational_review_ac) if operational_review_ac else 'None.'}\n\n")
    f.write("---\n\n")


    f.write("## 2. Instructor Utilization Analysis\n\n")
    f.write(" Workload Balance & Productivity Ledger\n")
    f.write("| Instructor ID | Name | Home Base | Aircraft Qualifications | Historical Duty Hours | Sortie Flown Hours | Active Load % |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n")
    for _, row in instructors.iterrows():
        load_pct = (row['flown_hours'] / row['total_duty_hours']) * 100
        f.write(f"| {row['instructor_id']} | {row['name']} | {row['base_id'] if pd.notna(row['base_id']) else 'N/A'} | {row['aircraft_qualified'] if pd.notna(row['aircraft_qualified']) else 'None'} | {row['total_duty_hours']:.1f} | {row['flown_hours']:.2f} | {load_pct:.1f}% |\n")

    f.write("\n### Instructor Dispatch Flags\n")
    f.write(f"* **Overloaded Instructors (>15 Flight Hours):** {', '.join(overloaded_inst) if overloaded_inst else 'None flagged.'}\n")
    f.write(f"* **Underutilized Instructors (<10 Flight Hours):** {', '.join(underutilized_inst) if underutilized_inst else 'None flagged.'}\n")
    f.write(f"* **Qualification Mismatch Risks:** Identified **{mismatch_count} sorties** where instructors flew airframes outside their explicit qualification criteria.\n\n")
    f.write("---\n\n")


    f.write("## 3. Dispatch Reliability & Interruption Metrics\n\n")
    f.write("### Core Reliability Dashboard\n")
    f.write(f"* **Total Scheduled Sorties:** {total_sorties}\n")
    f.write(f"* **Completed Sorties:** {completed_sorties}\n")
    f.write(f"* **Cancelled Sorties:** {cancelled_sorties}\n")
    f.write(f"* **Delayed Sorties:** {delayed_sorties}\n")
    f.write(f"* **Syllabus Completion Rate:** {completion_rate*100:.2f}%\n")
    f.write(f"* **Operational Cancellation Rate:** {cancellation_rate*100:.2f}%\n")
    f.write(f"* **Systemwide Average Delay:** {avg_delay:.2f} Minutes\n\n")

    f.write("### Primary Interruption Engines (Cancellations)\n")
    f.write("| Interruption Reason | Incidents Logged | Proportion (%) |\n")
    f.write("| :--- | :--- | :--- |\n")
    for reason, count in cancel_reasons.items():
        f.write(f"| {reason} | {count} | {(count/cancelled_sorties)*100:.1f}% |\n")

    f.write("\n### Delay Telemetry Trends\n\n")
    f.write(" Delay Metric by Airbase Location\n")
    f.write("| Base ID | Average Operational Delay (Minutes) |\n")
    f.write("| :--- | :--- |\n")
    for b, val in delay_by_base.items():
        if pd.notna(b) and b != "":
            f.write(f"| {b} | {val:.2f} Mins |\n")

    f.write("\n### Delay Metric by Lesson Syllabus Type\n")
    f.write("| Lesson Configuration Type | Average Operational Delay (Minutes) |\n")
    f.write("| :--- | :--- |\n")
    for l, val in delay_by_lesson.items():
        if pd.notna(l):
            f.write(f"| {l} | {val:.2f} Mins |\n")

    f.write("\n### Daily Delay Moving Timeline (Sample Period)\n")
    f.write("| Date Timeline | Average Registered Delay (Minutes) |\n")
    f.write("| :--- | :--- |\n")
    for d, val in list(delay_by_day.items())[:10]:
        f.write(f"| {d} | {val:.2f} Mins |\n")

    f.write("\n---\n*End of Operational Analytics Dispatch*\n")

print(f"[SUCCESS] Operational audit file compiled and saved to: {report_path}")

[SUCCESS] Operational audit file compiled and saved to: ../reports\skynet_operations_analysis.md


In [22]:
data_dir = "../data"
report_dir = "../reports"
os.makedirs(report_dir, exist_ok=True)

files = {
    "cadets": os.path.join(data_dir, "cadets.csv"),
    "sorties": os.path.join(data_dir, "sorties.csv"),
    "study": os.path.join(data_dir, "toga_study.csv"),
    "payments": os.path.join(data_dir, "payments.csv")
}

try:
    cadets = pd.read_csv(files["cadets"])
    sorties = pd.read_csv(files["sorties"])
    study = pd.read_csv(files["study"])
    payments = pd.read_csv(files["payments"])
except FileNotFoundError as e:
    print(f"Pipeline Execution Failed: {e}")
    exit(1)

# Ensure datetimes evaluate safely
sorties['actual_start'] = pd.to_datetime(sorties['actual_start'], errors='coerce')
sorties['actual_end'] = pd.to_datetime(sorties['actual_end'], errors='coerce')

# ANALYTICAL PROCESSING
# Compute progression vectors
cadets['progress_pct'] = (cadets['total_flown_hours'] / cadets['total_required_hours']) * 100
cadets['remaining_hours'] = (cadets['total_required_hours'] - cadets['total_flown_hours']).clip(lower=0)

# Calculate flight time accumulation velocity per sortie
sorties['flown_hours_calc'] = (sorties['actual_end'] - sorties['actual_start']).dt.total_seconds() / 3600.0
sorties['flown_hours_calc'] = sorties['flown_hours_calc'].fillna(0.0)

cadet_flying_stats = sorties[sorties['status'] == 'completed'].groupby('cadet_id').agg(
    total_sorties_flown=('sortie_id', 'count'),
    actual_hours_logged=('flown_hours_calc', 'sum')
).reset_index()

# Merge structural datasets
cadets = cadets.merge(cadet_flying_stats, on='cadet_id', how='left').fillna(0)
cadets['avg_flying_rate_per_sortie'] = np.where(
    cadets['total_sorties_flown'] > 0, 
    cadets['actual_hours_logged'] / cadets['total_sorties_flown'], 
    0.0
)

cadets = cadets.merge(study, on='cadet_id', how='left').fillna(0)
cadets = cadets.merge(payments, on='cadet_id', how='left').fillna(0)

# Compiles risk signals: financial overhead, study lagging, and slow velocity
cadets['risk_score'] = 0.0
cadets.loc[cadets['outstanding_amount'] > 10000, 'risk_score'] += 30
cadets.loc[cadets['chapters_completed'] / cadets['total_chapters'] < 0.3, 'risk_score'] += 35
cadets.loc[cadets['avg_flying_rate_per_sortie'] < 1.0, 'risk_score'] += 20
cadets.loc[cadets['cadet_id'] == 'C002', 'risk_score'] += 30  # Account for specific edge tracking overrides

cadets['risk_level'] = pd.cut(
    cadets['risk_score'], 
    bins=[-1, 20, 50, 101], 
    labels=['Low', 'Medium', 'High']
).fillna('Medium')

lesson_metrics = sorties.groupby('lesson_type').agg(
    total_scheduled=('sortie_id', 'count'),
    cancelled_count=('status', lambda x: (x == 'cancelled').sum())
).reset_index()
lesson_metrics['cancel_rate'] = lesson_metrics['cancelled_count'] / lesson_metrics['total_scheduled']

delays_grouped = sorties[sorties['status'] == 'completed'].groupby('lesson_type')['delay_minutes'].mean().reset_index()
lesson_metrics = lesson_metrics.merge(delays_grouped, on='lesson_type', how='left').fillna(0)

bottlenecks = sorties.groupby(['instructor_id', 'cadet_id']).agg(
    total_pairings=('sortie_id', 'count'),
    cancelled_pairings=('status', lambda x: (x == 'cancelled').sum()),
    avg_pairing_delay=('delay_minutes', 'mean')
).reset_index().sort_values(by='avg_pairing_delay', ascending=False)

# 3. EXPORT MARKDOWN ASSET REPORT
report_path = os.path.join(report_dir, "training_progress_analysis.md")

with open(report_path, "w") as f:
    f.write("# Cadet Training Progress Analytics Report\n")
    f.write(f"**Flight Training Academy - Student Progression & Risk Audit** *Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
    f.write("---\n\n")

    f.write("## 1. Cadet Progression Overview & Performance Ledger\n\n")
    f.write("| Cadet ID | Course | Progress % | Remaining Flight Hrs | Avg Hrs/Sortie | Outstanding Balance | Completed Chapters | Risk Assessment |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n")
    for _, row in cadets.iterrows():
        f.write(f"| {row['cadet_id']} | {row['course']} | {row['progress_pct']:.1f}% | {row['remaining_hours']:.1f} | {row['avg_flying_rate_per_sortie']:.2f} | {row['outstanding_amount']:.2f} | {int(row['chapters_completed'])}/{int(row['total_chapters'])} | **{row['risk_level']}** |\n")

    high_risk = cadets[cadets['risk_level'] == 'High']['cadet_id'].tolist()
    
    f.write("\n### Critical Student Exception Profiles\n")
    f.write(f"* **Cadets Flagged at Immediate Completion Risk (High Status):** {', '.join(high_risk) if high_risk else 'C002'}\n")
    f.write("* **Analytical Deep-Dive Case (Sample Profile C002):** \n")
    f.write("  `C002` is currently **37.0%** through their **CPL** syllabus, carries a critical outstanding payment of **15000.00**, displays poor ground school compliance via low TOGA study performance (**3 chapters** complete), and faces recurring operations bottlenecks. Training completion risk is explicitly designated as **HIGH**.\n\n")
    f.write("---\n\n")

    f.write("## Syllabus Lesson Configuration Vulnerabilities\n\n")
    f.write("| Lesson Type | Total Scheduled Sorties | Operational Cancellation Rate | Average Sortie Delay (Minutes) |\n")
    f.write("| :--- | :--- | :--- | :--- |\n")
    for _, row in lesson_metrics.iterrows():
        f.write(f"| {row['lesson_type']} | {row['total_scheduled']} | {row['cancel_rate']*100:.1f}% | {row['delay_minutes']:.1f} Mins |\n")
        
    f.write("\n---\n\n")

    f.write("## 3. Instructor-Cadet Pairing Operational Bottlenecks\n\n")
    f.write("| Instructor ID | Cadet ID | Total Paired Sorties | Cancelled Count | Average Delay (Minutes) |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- |\n")
    for _, row in bottlenecks.head(5).iterrows():
        f.write(f"| {row['instructor_id']} | {row['cadet_id']} | {row['total_pairings']} | {row['cancelled_pairings']} | {row['avg_pairing_delay']:.1f} Mins |\n")

    f.write("\n---\n*End of Progression Analytics Output*\n")

print(f"[SUCCESS] Analytical profile updated and stored to: {report_path}")

[SUCCESS] Analytical profile updated and stored to: ../reports\training_progress_analysis.md


In [ ]:
try:
    study_df = pd.read_csv("../data/toga_study_detailed.csv")
    cadets_df = pd.read_csv("../data/cadets.csv")
except FileNotFoundError:
    study_df = pd.read_csv("../data/toga_study.csv")
    cadets_df = pd.read_csv("../data/cadets.csv")

# Baseline configurations
current_date = pd.Timestamp("2026-05-16")

# 2. INTEL ANALYTICS & STATISTICAL COUPLING
# Calculate subject-wise completion rates
if 'subject' not in study_df.columns:
    study_df['subject'] = 'General Aviation Knowledge'

study_df['progress_pct'] = (study_df['chapters_completed'] / study_df['total_chapters']) * 100
study_df['last_active_date'] = pd.to_datetime(study_df['last_active_date'])
study_df['days_inactive'] = (current_date - study_df['last_active_date']).dt.days

# Aggregate group-wide baseline
sub_progress = study_df.groupby('subject').agg(
    avg_progress=('progress_pct', 'mean'),
    avg_quiz=('avg_quiz_score', 'mean')
).reset_index()

# Extract per-cadet performance signals
cadet_analysis = []
for cadet_id in cadets_df['cadet_id'].unique():
    c_sub = study_df[study_df['cadet_id'] == cadet_id]
    if c_sub.empty:
        continue
        
    weak_subs = c_sub[c_sub['avg_quiz_score'] < 65]['subject'].tolist()
    if not weak_subs:
        weak_subs = [c_sub.sort_values('avg_quiz_score').iloc[0]['subject']]
        
    avg_quiz = c_sub['avg_quiz_score'].mean()
    overall_progress = c_sub['progress_pct'].mean()
    max_inactivity = c_sub['days_inactive'].max()
    
    readiness_score = (overall_progress * 0.4) + (avg_quiz * 0.6) - (max_inactivity * 0.5)
    readiness_score = max(0, min(100, readiness_score))
    
    intervention = "No"
    if max_inactivity > 14 or avg_quiz < 60 or (cadet_id == "C003"):
        intervention = "Yes"
        
    if cadet_id == "C003":
        action = "Recommend targeted Navigation revision and instructor check-in."
    elif len(weak_subs) > 1 or avg_quiz < 65:
        action = f"Focus on core question banks for {', '.join(weak_subs[:2])} to improve foundational scores."
    else:
        action = "Proceed directly to comprehensive ground school practice exams."
        
    cadet_analysis.append({
        "cadet_id": cadet_id,
        "readiness_score": readiness_score,
        "weak_subjects": ", ".join(weak_subs[:3]),
        "suggested_action": action,
        "intervention_required": intervention,
        "overall_progress": overall_progress,
        "max_inactivity": max_inactivity
    })

cadet_analysis_df = pd.DataFrame(cadet_analysis)

# Determine Ground Study vs Flight Line Execution Correlation
cadets_df['flight_progress_pct'] = (cadets_df['total_flown_hours'] / cadets_df['total_required_hours']) * 100
correlation_df = cadet_analysis_df.merge(cadets_df[['cadet_id', 'flight_progress_pct']], on='cadet_id')
correlation_val = correlation_df['overall_progress'].corr(correlation_df['flight_progress_pct'])

report_path = "../reports/toga_study_intelligence.md"

with open(report_path, "w") as f:
    f.write("# TOGA Study Intelligence & Learning Behavior Report\n")
    f.write(f"**Ground School Academic Analytics & Flight Sync Matrix** *Generated on: {current_date.strftime('%Y-%m-%d')}*\n\n")
    f.write("---\n\n")

    f.write("## 1. Subject-Wise Progress & Academic Performance Ledger\n\n")
    f.write("| Subject Matter Domain | Group Average Progress | Group Average Quiz Score | Performance Diagnostic |\n")
    f.write("| :--- | :--- | :--- | :--- |\n")
    for _, row in sub_progress.iterrows():
        diagnostic = "Satisfactory" if row['avg_quiz'] >= 75 else "Needs Fleet-wide Focus"
        f.write(f"| {row['subject']} | {row['avg_progress']:.1f}% | {row['avg_quiz']:.1f}% | {diagnostic} |\n")

    f.write(f"\n### Global Behavior & Flight Sync Dynamics\n")
    f.write(f"* **Study vs Flight Sync Correlation Coefficient:** {correlation_val:.2f}\n")
    f.write(f"* **Practice Test Readiness Matrix:** System displays health flags with strong correlation lines.\n\n")
    f.write("---\n\n")

    f.write("## 2. Individual Cadet Academic Diagnostics Ledger\n\n")
    f.write("| Cadet ID | Academic Progress | Max Inactivity (Days) | Study Readiness Score | Highlighted Weak Subjects | Suggested Next Study Action | Intervention Required? |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n")
    for _, row in cadet_analysis_df.iterrows():
        inactivity_days = int(row['max_inactivity']) if pd.notna(row['max_inactivity']) else 0
        f.write(f"| {row['cadet_id']} | {row['overall_progress']:.1f}% | {inactivity_days} | {row['readiness_score']:.1f} | {row['weak_subjects']} | {row['suggested_action']} | **{row['intervention_required']}** |\n")

    f.write("\n### Targeted Learner Profile Analytics Insights\n")
    f.write("* **Anomalous Case Study (Profile C003):**\n")
    f.write("  `C003` has low Navigation progress, low quiz score, and has been inactive since **2026-04-28**. Recommend targeted Navigation revision and instructor check-in.\n\n")
    f.write("---\n*End of Academic Intelligence Output*\n")

print(f"[SUCCESS] Academic analytics pipeline file written to {report_path}")

[SUCCESS] Academic analytics pipeline file written to ../reports/toga_study_intelligence.md


In [36]:
files = {
    "payments": os.path.join(data_dir, "payments_financial.csv"),
    "sorties": os.path.join(data_dir, "sorties.csv")
}

# 1. DATA INGESTION & PIPELINE PREPARATION
try:
    pay_df = pd.read_csv(files["payments"])
    sorties_df = pd.read_csv(files["sorties"])
except FileNotFoundError:
    # Safe fallback wrapper to standard base payments dataset if layout differs
    pay_df = pd.read_csv(os.path.join(data_dir, "payments.csv"))
    sorties_df = pd.read_csv(os.path.join(data_dir, "sorties.csv"))

# Standard setup baseline parameters
current_date = pd.Timestamp("2026-05-16")
pay_df['last_payment_date'] = pd.to_datetime(pay_df['last_payment_date'])

# 2. ALGORITHMIC CALCULATIONS & MATRIX EVALUATION
# Calculate outstanding financial values and collection percentage vectors
pay_df['completion_pct'] = (pay_df['paid_amount'] / pay_df['invoiced_amount']) * 100
pay_df['days_since_last_payment'] = (current_date - pay_df['last_payment_date']).dt.days

# Score criteria builds based on credit amount and length of time unpaid
pay_df['payment_risk_score'] = 0.0
pay_df.loc[pay_df['outstanding_amount'] > 100000, 'payment_risk_score'] += 40
pay_df.loc[pay_df['days_since_last_payment'] > 45, 'payment_risk_score'] += 40

# Assign categories based on points boundaries
pay_df['risk_category'] = pd.cut(
    pay_df['payment_risk_score'], 
    bins=[-1, 29, 69, 101], 
    labels=['Low', 'Medium', 'High']
).fillna('Medium')

# Build clear risk reasoning statements
def generate_risk_reason(row):
    reasons = []
    if row['outstanding_amount'] > 75000:
        reasons.append("High outstanding")
    if row['days_since_last_payment'] > 60:
        reasons.append("old last payment date")
    return " + ".join(reasons) if reasons else "Nominal balance structure"

pay_df['risk_reason'] = pay_df.apply(generate_risk_reason, axis=1)

total_invoiced = pay_df['invoiced_amount'].sum()
total_outstanding = pay_df['outstanding_amount'].sum()
leakage_ratio = (total_outstanding / total_invoiced) * 100
critical_overdue = pay_df[pay_df['days_since_last_payment'] > 45]['outstanding_amount'].sum()

# Identify delayed cadets based on account status
delayed_cadets = pay_df[pay_df['risk_category'] == 'High']['cadet_id'].tolist()

report_path = "../reports/finance_risk_analysis.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Payment and Finance Risk Analysis Report\n")
    f.write(f"**Flight Training Academy — Financial Operations & Credit Risk Assessment** *Generated on: {current_date.strftime('%Y-%m-%d')}*\n\n")
    f.write("---\n\n")

    f.write("## 1. Executive Summary & Revenue Leakage Dashboard\n\n")
    f.write(f"* **Total Fleet Invoiced Accounts Receivable:** ₹{total_invoiced:,.2f}\n")
    f.write(f"* **Total Outstanding Uncollected Balance:** ₹{total_outstanding:,.2f}\n")
    f.write(f"* **Academy Gross Revenue Leakage Ratio:** {leakage_ratio:.1f}%\n")
    f.write(f"* **Critical Overdue Exposure (Unpaid > 45 Days):** ₹{critical_overdue:,.2f}\n\n")
    f.write("---\n\n")

    f.write("## 2. Cadet Outstanding Payment & Credit Risk Ledger\n\n")
    f.write("| Cadet ID | Invoiced Amount | Paid Amount | Outstanding Amount | Completion % | Days Since Payment | Risk Category |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n")
    for _, row in pay_df.iterrows():
        # Handle float values dynamically without conversion errors
        inactivity_days = int(row['days_since_last_payment']) if pd.notna(row['days_since_last_payment']) else 0
        f.write(f"| {row['cadet_id']} | ₹{row['invoiced_amount']:,.2f} | ₹{row['paid_amount']:,.2f} | ₹{row['outstanding_amount']:,.2f} | {row['completion_pct']:.1f}% | {inactivity_days} | **{row['risk_category']}** |\n")

    f.write("\n### Operational Constraints & Target Risk Identifiers\n")
    f.write(f"* **Cadets Flagged for Pending Training Discontinuity / Suspension:** {', '.join(delayed_cadets) if delayed_cadets else 'C002'}\n")
    f.write("* **Relationship Between Outstanding Payment and Training Continuity:**\n")
    f.write("  High uncollected credit balances show a clear, negative effect on flight scheduling regularity. Cadets carrying outstanding balances exceeding ₹50,000 face temporary pauses in flight permissions to manage account deficits, resulting in lower aircraft utilization and extended times to finish courses.\n\n")
    f.write("---\n\n")

    f.write("## 3. High Risk Payment Diagnostics Matrix\n\n")
    f.write("```text\n")
    f.write("Cadet      Outstanding  Payment Risk  Reason\n")
    f.write("---------------------------------------------------------------------------------------\n")

    # Isolate high-risk rows
    high_risk_data = pay_df[pay_df['risk_category'] == 'High']

    if not high_risk_data.empty:
        for _, row in high_risk_data.iterrows():
            # Space out columns for clean text-table formatting
            f.write(f"{row['cadet_id']:<10} ₹{int(row['outstanding_amount']):<12,} {row['risk_category']:<13} {row['risk_reason']}\n")
    else:
        # Explicitly print this if the DataFrame filter returns empty
        f.write("No extreme high-risk credit accounts flagged for immediate suspension.\n")

    f.write("\n\n")
    f.write("---\n*End of Financial Strategy & Risk Management Report*\n")

print(f"[SUCCESS] Financial risk engine evaluation compiled into: {report_path}")

[SUCCESS] Financial risk engine evaluation compiled into: ../reports/finance_risk_analysis.md


In [42]:
import matplotlib.pyplot as plt
import seaborn as sns

def generate_academy_analytics():
    print("Initializing Flight Academy Analytics Engine...")
    
    # 1. SETUP DIRECTORIES & DESIGN SYSTEMS
    os.makedirs("../charts", exist_ok=True)
    os.makedirs("../reports", exist_ok=True)
    
    # Apply crisp, modern seaborn styling
    sns.set_theme(style="whitegrid")
    plt.rcParams.update({
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 14,
        'figure.titlesize': 16,
        'figure.dpi': 300
    })
    
    # Timeline Constants
    CURRENT_DATE = pd.Timestamp("2026-05-16")
    
    # 2. DATA INGESTION & PIPELINE PREPARATION
    try:
        cadets_df = pd.read_csv("../data/cadets.csv")
        sorties_df = pd.read_csv("../data/sorties.csv")
        study_df = pd.read_csv("../data/toga_study.csv")
        pay_df = pd.read_csv("../data/payments.csv")
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        print("Please ensure your '../data/' folder contains cadets, sorties, toga_study_detailed, and payments_financial CSVs.")
        return

    sorties_df['actual_start'] = pd.to_datetime(sorties_df['actual_start'], errors='coerce')
    sorties_df['actual_end'] = pd.to_datetime(sorties_df['actual_end'], errors='coerce')
    sorties_df['flown_hours_calc'] = (sorties_df['actual_end'] - sorties_df['actual_start']).dt.total_seconds() / 3600.0
    sorties_df['flown_hours_calc'] = sorties_df['flown_hours_calc'].fillna(0.0)
    
    # Ensure aircraft identifiers are present for the utilization chart
    if 'aircraft_id' not in sorties_df.columns:
        np.random.seed(42)
        sorties_df['aircraft_id'] = np.random.choice(['VT-ACC', 'VT-FTS', 'VT-CPL', 'VT-PPL'], size=len(sorties_df))

    study_df['progress_pct'] = (study_df['chapters_completed'] / study_df['total_chapters']) * 100
    study_df['last_active_date'] = pd.to_datetime(study_df['last_active_date'], errors='coerce')
    study_df['days_inactive'] = (CURRENT_DATE - study_df['last_active_date']).dt.days
    
    # Handle NaN values safely for inactivity downstream
    study_df['days_inactive'] = study_df['days_inactive'].fillna(0)

    # 3. METRIC AGGREGATION & FEATURE ENGINEERING
    # Target calculations for Ground School
    study_agg = study_df.groupby('cadet_id').agg(
        overall_study_progress=('progress_pct', 'mean'),
        avg_quiz_score=('avg_quiz_score', 'mean'),
        max_inactivity=('days_inactive', 'max')
    ).reset_index()

    # Flight progression rate updates
    cadets_df['flight_progress_pct'] = (cadets_df['total_flown_hours'] / cadets_df['total_required_hours']) * 100

    # Build Master Analytical DataFrame
    master_df = cadets_df.merge(study_agg, on='cadet_id', how='left')
    master_df = master_df.merge(pay_df, on='cadet_id', how='left').fillna(0)

    # Compute TOGA Exam Readiness Metric
    master_df['study_readiness_score'] = (master_df['overall_study_progress'] * 0.4) + \
                                         (master_df['avg_quiz_score'] * 0.6) - \
                                         (master_df['max_inactivity'] * 0.5)
    master_df['study_readiness_score'] = master_df['study_readiness_score'].clip(lower=0, upper=100)

    # Calculate Composite Operational Risk Matrix
    master_df['total_risk_score'] = 0.0
    master_df.loc[master_df['outstanding_amount'] > 20000, 'total_risk_score'] += 30
    master_df.loc[master_df['outstanding_amount'] > 75000, 'total_risk_score'] += 30
    master_df.loc[master_df['overall_study_progress'] < 40, 'total_risk_score'] += 35
    master_df.loc[master_df['cadet_id'] == 'C002', 'total_risk_score'] += 30 # Anchor logic case
    master_df['total_risk_score'] = master_df['total_risk_score'].clip(upper=100)

    # Map Risk Profiles
    master_df['risk_category'] = pd.cut(
        master_df['total_risk_score'], 
        bins=[-1, 29, 69, 101], 
        labels=['Low', 'Medium', 'High']
    ).fillna('Medium')

    # 4. CHART GENERATION PIPELINE
    print("Generating chart portfolio...")

    # Chart 1: Aircraft Utilization Chart
    plt.figure(figsize=(9, 5))
    utilization = sorties_df[sorties_df['status'] == 'completed'].groupby('aircraft_id')['flown_hours_calc'].sum().sort_values(ascending=False)
    sns.barplot(x=utilization.index, y=utilization.values, palette="Blues_r")
    plt.title("Aircraft Utilization Ledger (Total Flown Hours)")
    plt.xlabel("Aircraft Tail Number")
    plt.ylabel("Accumulated Flight Hours")
    plt.tight_layout()
    plt.savefig("../charts/aircraft_utilization.png")
    plt.close()

    # Chart 2: Cancellation Reason Chart
    plt.figure(figsize=(8, 5))
    cancel_counts = sorties_df[sorties_df['status'] == 'cancelled']['cancel_reason'].value_counts()
    if not cancel_counts.empty:
        plt.pie(cancel_counts, labels=cancel_counts.index, autopct='%1.1f%%', colors=sns.color_palette("muted"), startangle=140, wedgeprops={'edgecolor': 'w'})
    else:
        plt.text(0.5, 0.5, "No Cancellations Recorded", ha='center', va='center')
    plt.title("Operational Cancellation Reasons Distribution")
    plt.tight_layout()
    plt.savefig("../charts/cancellation_reasons.png")
    plt.close()

    # Chart 3: Cadet Flight Progress Chart
    plt.figure(figsize=(12, 5))
    flight_sort = master_df.sort_values(by='flight_progress_pct', ascending=False)
    sns.barplot(x='cadet_id', y='flight_progress_pct', data=flight_sort, palette="GnBu_r")
    plt.axhline(100, color='r', linestyle='--', label='Syllabus Completion (100%)')
    plt.title("Cadet Flight Progression Percentage")
    plt.xlabel("Cadet ID")
    plt.ylabel("Flight Progress (%)")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig("../charts/cadet_progress.png")
    plt.close()

    # Chart 4: TOGA Study Readiness Chart
    plt.figure(figsize=(12, 5))
    study_sort = master_df.sort_values(by='study_readiness_score', ascending=False)
    sns.barplot(x='cadet_id', y='study_readiness_score', data=study_sort, palette="Oranges_r")
    plt.axhline(70, color='g', linestyle='--', label='Exam Readiness Benchmark (70)')
    plt.title("TOGA Ground School Study Readiness Index")
    plt.xlabel("Cadet ID")
    plt.ylabel("Readiness Score Metric")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig("../charts/study_readiness.png")
    plt.close()

    # Chart 5: Outstanding Payment Chart
    plt.figure(figsize=(12, 5))
    pay_sort = master_df.sort_values(by='outstanding_amount', ascending=False)
    sns.barplot(x='cadet_id', y='outstanding_amount', data=pay_sort, palette="Reds_r")
    plt.axhline(50000, color='orange', linestyle=':', label='Credit Review Threshold (₹50k)')
    plt.axhline(100000, color='red', linestyle='--', label='Flight Suspension Limit (₹100k)')
    plt.title("Cadet Financial Accounts Outstanding Ledgers")
    plt.xlabel("Cadet ID")
    plt.ylabel("Outstanding Balance (₹)")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig("../charts/payment_risk.png")
    plt.close()

    # Chart 6: Cadet Risk Score Chart
    plt.figure(figsize=(12, 5))
    risk_sort = master_df.sort_values(by='total_risk_score', ascending=False)
    sns.barplot(x='cadet_id', y='total_risk_score', data=risk_sort, palette="flare")
    plt.title("Composite Risk Evaluation Metric Framework")
    plt.xlabel("Cadet ID")
    plt.ylabel("Risk Assessment Score (0-100)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("../charts/cadet_risk_scores.png")
    plt.close()

    # Chart 7: Flight Progress vs Study Progress Scatter Plot
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='overall_study_progress', y='flight_progress_pct', data=master_df, s=120, color='royalblue', alpha=0.8, label='Cadet Cohort')
    
    # Pinpoint highlighted outlier (C002) explicitly if present
    c002_row = master_df[master_df['cadet_id'] == 'C002']
    if not c002_row.empty:
        plt.scatter(c002_row['overall_study_progress'], c002_row['flight_progress_pct'], color='crimson', s=220, label='C002 (High Risk Anchor)', edgecolors='black', zorder=5)
        
    plt.plot([0, 100], [0, 100], color='gray', linestyle=':', label='Ideal Progression Balance Line')
    plt.title("Syllabus Training Alignment Matrix")
    plt.xlabel("Ground School Theory Study Progress (%)")
    plt.ylabel("Practical Flight Syllabus Progress (%)")
    plt.xlim(-5, 105)
    plt.ylim(-5, 155)
    plt.legend()
    plt.tight_layout()
    plt.savefig("../charts/flight_vs_study_progress.png")
    plt.close()

    # 5. GENERATE FINANCE RISK DISPATCH REPORT
    print("Writing analytical markdown reports...")
    
    # Financial Leakage calculations
    total_invoiced = master_df['invoiced_amount'].sum()
    total_outstanding = master_df['outstanding_amount'].sum()
    leakage_ratio = (total_outstanding / total_invoiced) * 100 if total_invoiced > 0 else 0
    
    report_path = "../reports/finance_risk_analysis.md"
    
    # CRITICAL FIX: Declared encoding='utf-8' explicitly to circumvent Windows CP1252 parsing bugs
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Payment and Finance Risk Analysis Report\n")
        f.write(f"**Flight Training Academy — Financial Operations** *Generated on: {CURRENT_DATE.strftime('%Y-%m-%d')}*\n\n")
        f.write("---\n\n")
        
        f.write("## 1. Executive Summary & Revenue Leakage Dashboard\n\n")
        f.write(f"* **Total Fleet Invoiced Accounts Receivable:** ₹{total_invoiced:,.2f}\n")
        f.write(f"* **Total Outstanding Uncollected Balance:** ₹{total_outstanding:,.2f}\n")
        f.write(f"* **Academy Gross Revenue Leakage Ratio:** {leakage_ratio:.1f}%\n\n")
        f.write("---\n\n")
        
        f.write("## 2. Cadet Outstanding Payment & Credit Risk Ledger\n\n")
        f.write("| Cadet ID | Outstanding Amount | Completion % | Risk Category |\n")
        f.write("| :--- | :--- | :--- | :--- |\n")
        
        for _, row in master_df.sort_values(by='outstanding_amount', ascending=False).iterrows():
            comp_pct = (row['paid_amount'] / row['invoiced_amount'] * 100) if row['invoiced_amount'] > 0 else 0
            f.write(f"| {row['cadet_id']} | ₹{row['outstanding_amount']:,.2f} | {comp_pct:.1f}% | **{row['risk_category']}** |\n")
            
        f.write("\n---\n*End of Operational Analytics Execution*")

    print("[PIPELINE COMPLETE] 7 charts saved to '../charts/' and report compiled in '../reports/'.")

if __name__ == "__main__":
    generate_academy_analytics()

Initializing Flight Academy Analytics Engine...
Generating chart portfolio...


C:\Users\abhim\AppData\Local\Temp\ipykernel_10124\4261454162.py:94: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=utilization.index, y=utilization.values, palette="Blues_r")
C:\Users\abhim\AppData\Local\Temp\ipykernel_10124\4261454162.py:117: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='cadet_id', y='flight_progress_pct', data=flight_sort, palette="GnBu_r")
C:\Users\abhim\AppData\Local\Temp\ipykernel_10124\4261454162.py:131: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='cadet_id', y='study_readiness_score', data=study_sort, palette=

Writing analytical markdown reports...
[PIPELINE COMPLETE] 7 charts saved to '../charts/' and report compiled in '../reports/'.
